In [1]:
import os
import json
import yaml
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch
import re
from tqdm import tqdm
from pathlib import Path

/Users/karunachandra/anaconda3/envs/safe_ai/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import judgezoo
from judgezoo import Judge
judge = Judge.from_name("strong_reject")

Loading weights: 100%|██████████| 252/252 [00:00<00:00, 1574.97it/s]


In [3]:
def get_judge_evals(questions, responses):
    judgezoo_scores = []
    for question, response in zip(questions, responses):
        conversation = [
            {"role": "user", "content": question},
            {"role": "assistant", "content": response}
        ]
        judgezoo_scores.append(judge([conversation]))
    return judgezoo_scores

In [4]:
output_dir = "downloaded_outputs/outputs_with_judgezoo_scores"

In [6]:
for file in tqdm(os.listdir('/Users/karunachandra/Documents/Tuebingen/SS26/AIsafety/assignment/MiniProject2/task1/downloaded_outputs/outputs')):
    if file.endswith('.json'):
        with open(os.path.join('/Users/karunachandra/Documents/Tuebingen/SS26/AIsafety/assignment/MiniProject2/task1/downloaded_outputs/outputs', file), 'r') as f:
            data = json.load(f)
        
        questions = []
        responses = []
        for id, data_vals in data.items():
            questions.append(data_vals['question'])
            responses.append(data_vals['response'])

        judgezoo_scores = get_judge_evals(questions, responses)

        updated_data = {}
        for id, data_vals in data.items():
            index= list(data.keys()).index(id)
            updated_data[id] = {
                'question': data_vals['question'],
                'response': data_vals['response'],
                'judgezoo_score': judgezoo_scores[index],
                'coherence_score': data_vals['coherence_score'],
                'align_score': data_vals['align_score']
            }
        
        output_path = os.path.join(output_dir, file)
        with open(output_path, 'w') as f:
            json.dump(updated_data, f, indent=4)

100%|██████████| 160/160 [41:51<00:00, 15.70s/it]
